In [78]:
# Add these imports at the top of your file
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm  # Add this import
import rasterio
import heapq
import time



In [79]:
class TerrainTraverser:
    def __init__(self, slope_penalty_factor=5.0):
        self.slope_penalty_factor = slope_penalty_factor
        self.traversability_scores = None
        
    def calculate_traversability(self, grid):
        height, width = grid.shape[0], grid.shape[1]
        self.traversability_scores = np.ones((height, width)) * 100
        
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
        
        for x in range(height):
            for y in range(width):
                max_slope = 0
                for dx, dy in directions:
                    nx, ny = x + dx, y + dy
                    if (0 <= nx < height and 0 <= ny < width):
                        elev1 = grid[x, y, 1]
                        elev2 = grid[nx, ny, 1]
                        elevation_diff = abs(elev2 - elev1)
                        
                        distance = np.sqrt(dx*dx + dy*dy)
                        slope = np.degrees(np.arctan2(elevation_diff, distance))
                        max_slope = max(max_slope, slope)
                
                if max_slope < 15:
                    score = 100 - (max_slope / 15) * 20
                elif max_slope < 45:
                    score = 80 - ((max_slope - 15) / 30) * 40
                else:
                    score = 40 - ((max_slope - 45) / 45) * 40
                    
                self.traversability_scores[x, y] = max(0, min(100, score))
        
        return self.traversability_scores

    def get_neighbors(self, pos, grid):
        neighbors = []
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1)]
        
        for dx, dy in directions:
            new_x, new_y = pos[0] + dx, pos[1] + dy
            if not (0 <= new_x < grid.shape[0] and 0 <= new_y < grid.shape[1]):
                continue
            
            current_elevation = grid[pos[0], pos[1], 1]
            new_elevation = grid[new_x, new_y, 1]
            elevation_diff = abs(new_elevation - current_elevation)
            
            distance = np.sqrt(dx*dx + dy*dy)
            slope = np.degrees(np.arctan2(elevation_diff, distance))
            
            traversability = self.traversability_scores[new_x, new_y]
            neighbors.append((new_x, new_y, traversability, slope))
                
        return neighbors

    def heuristic(self, a, b):
        return np.sqrt((b[0] - a[0])**2 + (b[1] - a[1])**2)

    def find_path(self, start, goal, grid, use_elevation_costs=True):
        start_time = time.time()
        self.calculate_traversability(grid)
        
        frontier = []
        heapq.heappush(frontier, (0, start))
        came_from = {start: None}
        cost_so_far = {start: 0}
        cell_slopes = {start: 0.0}
        
        max_steps = min(50000, grid.shape[0] * grid.shape[1])
        steps = 0
        
        while frontier and steps < max_steps:
            steps += 1
            current = heapq.heappop(frontier)[1]
            
            if current == goal:
                break
                
            for next_info in self.get_neighbors(current, grid):
                next_pos = (next_info[0], next_info[1])
                traversability = next_info[2]
                slope = next_info[3]
                
                if use_elevation_costs:
                    dx = next_pos[0] - current[0]
                    dy = next_pos[1] - current[1]
                    distance = np.sqrt(dx*dx + dy*dy)
                    
                    current_elevation = grid[current[0], current[1], 1]
                    new_elevation = grid[next_pos[0], next_pos[1], 1]
                    elevation_diff = new_elevation - current_elevation
                    
                    if slope < 20:
                        slope_penalty = slope / 20.0
                    elif slope < 45:
                        slope_penalty = 1.0 + (slope - 20) / 10.0
                    else:
                        slope_penalty = 3.5 + (slope - 45) * 0.5
                    
                    if elevation_diff > 0:
                        direction_factor = 1.5
                    else:
                        direction_factor = 0.7
                    
                    movement_cost = distance * (1.0 + slope_penalty * direction_factor)
                else:
                    dx = next_pos[0] - current[0]
                    dy = next_pos[1] - current[1]
                    distance = np.sqrt(dx*dx + dy*dy)
                    movement_cost = distance
                
                new_cost = cost_so_far[current] + movement_cost
                
                if next_pos not in cost_so_far or new_cost < cost_so_far[next_pos]:
                    cost_so_far[next_pos] = new_cost
                    cell_slopes[next_pos] = slope
                    priority = new_cost + self.heuristic(next_pos, goal)
                    heapq.heappush(frontier, (priority, next_pos))
                    came_from[next_pos] = current
        
        if goal not in came_from:
            return None, {}, 0
            
        path = []
        current = goal
        while current is not None:
            path.append(current)
            current = came_from[current]
        path.reverse()
        
        path_traversability = 0
        path_length = 0
        elevation_gain = 0
        elevation_loss = 0
        avg_slope = 0
        max_slope = 0
        
        if path and len(path) > 1:
            traversability_sum = sum(self.traversability_scores[pos[0], pos[1]] for pos in path)
            path_traversability = traversability_sum / len(path)
            
            slope_sum = 0
            slope_count = 0
            
            for i in range(len(path) - 1):
                dx = path[i+1][0] - path[i][0]
                dy = path[i+1][1] - path[i][1]
                segment_length = np.sqrt(dx*dx + dy*dy)
                path_length += segment_length
                
                elev_change = grid[path[i+1][0], path[i+1][1], 1] - grid[path[i][0], path[i][1], 1]
                if elev_change > 0:
                    elevation_gain += elev_change
                else:
                    elevation_loss += abs(elev_change)
                
                segment_slope = np.degrees(np.arctan2(abs(elev_change), segment_length))
                slope_sum += segment_slope
                slope_count += 1
                max_slope = max(max_slope, segment_slope)
            
            if slope_count > 0:
                avg_slope = slope_sum / slope_count
        
        end_time = time.time()
        processing_time = end_time - start_time
        
        print(f"Path calculation took {processing_time:.2f} seconds")
        print(f"Path found with {len(path)} steps, length: {path_length:.1f} units")
        print(f"Average traversability: {path_traversability:.1f}/100")
        print(f"Average slope: {avg_slope:.1f}°, Maximum slope: {max_slope:.1f}°")
        print(f"Elevation gain: {elevation_gain:.1f}, loss: {elevation_loss:.1f}")
        
        path_stats = {
            'traversability': path_traversability,
            'length': path_length,
            'avg_slope': avg_slope,
            'max_slope': max_slope,
            'elevation_gain': elevation_gain,
            'elevation_loss': elevation_loss
        }
        
        return path, path_stats, processing_time

def load_terrain_data(dem_path, region=None, downsample_factor=20):
    with rasterio.open(dem_path) as dem_dataset:
        dem = dem_dataset.read(1,masked = True).astype(np.float32)
        dem = dem[::downsample_factor, ::downsample_factor]
    
    if region is not None:
        y_min, y_max, x_min, x_max = region
        y_min = max(0, y_min)
        y_max = min(dem.shape[0], y_max)
        x_min = max(0, x_min)
        x_max = min(dem.shape[1], x_max)
        dem = dem[y_min:y_max, x_min:x_max]
    
    return dem

def check_valid_coordinates(dem, x, y):
    """
    Check if the given x, y coordinates have valid elevation data (not NaN).
    
    Parameters:
    dem (numpy.ndarray): Digital elevation model
    x (int): X coordinate
    y (int): Y coordinate
    
    Returns:
    bool: True if the location has valid elevation data, False otherwise
    """
    # Check if coordinates are within the DEM bounds
    if not (0 <= y < dem.shape[0] and 0 <= x < dem.shape[1]):
        print(f"Coordinates ({x}, {y}) are out of bounds!")
        return False
    
    # Check if the elevation at the given coordinates is NaN
    if np.isnan(dem[y, x]):
        print(f"Warning: No valid elevation data at coordinates ({x}, {y})!")
        return False
    
    return True

def analyze_terrain_paths(dem_path, start_point=None, end_point=None, downsample_factor=20, buffer_size=50, slope_emphasis=5):
    # Fix for the invalid value warnings in masked array operations
    with rasterio.open(dem_path) as dem_dataset:
        # Read the DEM data
        dem_data = dem_dataset.read(1, masked=True)
        
        # Convert masked values to NaN explicitly before downsampling
        dem_data_filled = dem_data.filled(np.nan)
        
        # Now downsample after handling masked values
        full_dem = dem_data_filled[::downsample_factor, ::downsample_factor]
    
    # Create a mask for invalid/NaN values
    valid_mask = ~np.isnan(full_dem)
    
    terrain_height, terrain_width = full_dem.shape
    print(f"Full terrain dimensions: {terrain_height} x {terrain_width}")
    
    if start_point is None or end_point is None:
        print("\nPlease enter coordinates for start and end points.")
        print(f"Valid range: x (0-{terrain_width-1}), y (0-{terrain_height-1})")
        
        # For starting point
        if start_point is None:
            valid_start = False
            while not valid_start:
                try:
                    start_x = int(input("Start X coordinate: "))
                    start_y = int(input("Start Y coordinate: "))
                    
                    if check_valid_coordinates(full_dem, start_x, start_y):
                        start_point = (start_y, start_x)
                        valid_start = True
                    else:
                        print("Please choose different start coordinates.")
                except ValueError:
                    print("Invalid input! Please enter integer coordinates.")
        
        # For ending point
        if end_point is None:
            valid_end = False
            while not valid_end:
                try:
                    end_x = int(input("End X coordinate: "))
                    end_y = int(input("End Y coordinate: "))
                    
                    if check_valid_coordinates(full_dem, end_x, end_y):
                        end_point = (end_y, end_x)
                        valid_end = True
                    else:
                        print("Please choose different end coordinates.")
                except ValueError:
                    print("Invalid input! Please enter integer coordinates.")
    
    # Additional check for out-of-bounds points (fallback to default points)
    if (start_point[0] < 0 or start_point[0] >= terrain_height or
        start_point[1] < 0 or start_point[1] >= terrain_width or
        end_point[0] < 0 or end_point[0] >= terrain_height or
        end_point[1] < 0 or end_point[1] >= terrain_width or
        np.isnan(full_dem[start_point[0], start_point[1]]) or
        np.isnan(full_dem[end_point[0], end_point[1]])):
        print("Points out of bounds or in invalid terrain! Using default points.")
        
        # Find valid terrain coordinates for default points
        valid_y, valid_x = np.where(valid_mask)
        if len(valid_y) > 0 and len(valid_x) > 0:
            # Try to pick points from different parts of the valid terrain
            quarter_idx = len(valid_y) // 4
            three_quarter_idx = 3 * len(valid_y) // 4
            
            start_point = (valid_y[quarter_idx], valid_x[quarter_idx])
            end_point = (valid_y[three_quarter_idx], valid_x[three_quarter_idx])
        else:
            # Fallback to simple division if we can't find valid points
            start_point = (terrain_height // 4, terrain_width // 4)
            end_point = (3 * terrain_height // 4, 3 * terrain_width // 4)
    
    print(f"Using Start: ({start_point[1]}, {start_point[0]}), End: ({end_point[1]}, {end_point[0]})")
    
    y_min = max(0, min(start_point[0], end_point[0]) - buffer_size)
    y_max = min(terrain_height, max(start_point[0], end_point[0]) + buffer_size)
    x_min = max(0, min(start_point[1], end_point[1]) - buffer_size)
    x_max = min(terrain_width, max(start_point[1], end_point[1]) + buffer_size)
    
    region = (y_min, y_max, x_min, x_max)
    
    dem_region = full_dem[y_min:y_max, x_min:x_max]
    
    local_start = (start_point[0] - y_min, start_point[1] - x_min)
    local_end = (end_point[0] - y_min, end_point[1] - x_min)
    
    print(f"Region dimensions: {dem_region.shape[0]} x {dem_region.shape[1]}")
    print(f"Local start: {local_start}, Local end: {local_end}")
    
    region_height, region_width = dem_region.shape
    grid = np.zeros((region_height, region_width, 2))
    
    # Handle NaN values in the grid
    dem_region_filled = np.nan_to_num(dem_region, nan=0.0)
    grid[:, :, 1] = dem_region_filled
    
    slope_penalty_factor = slope_emphasis * 1.0
    traverser = TerrainTraverser(slope_penalty_factor=slope_penalty_factor)
    
    print("\nFinding distance-optimized path (minimal slope consideration)...")
    fast_path, fast_stats, fast_time = traverser.find_path(
        local_start, local_end, grid, use_elevation_costs=False)
    
    print("\nFinding slope-optimized path (emphasizes gentle slopes)...")
    optimal_path, optimal_stats, optimal_time = traverser.find_path(
        local_start, local_end, grid, use_elevation_costs=True)
    
    visualize_paths(
        dem_region, 
        fast_path=fast_path, 
        optimal_path=optimal_path, 
        local_start=local_start, 
        local_end=local_end,
        buffer_region=(y_min, y_max, x_min, x_max),
        original_start=start_point,
        original_end=end_point
    )
    
    return {
        'fast_path': fast_path,
        'fast_stats': fast_stats,
        'fast_time': fast_time,
        'optimal_path': optimal_path,
        'optimal_stats': optimal_stats,
        'optimal_time': optimal_time,
        'region': region,
        'grid': grid
    }

In [80]:
def visualize_paths(dem, slope_map=None, fast_path=None, optimal_path=None, 
                   local_start=None, local_end=None, buffer_region=None, 
                   original_start=None, original_end=None, 
                   save_path="terrain_paths"):
    # Create base file name without extension
    base_save_path = save_path.replace('.png', '') if save_path.endswith('.png') else save_path
    
    # Handle NaN values first - create a clean version of the DEM for processing
    dem_clean = np.nan_to_num(dem, nan=0.0)
    
    # ------------------- Slope Map Visualization -------------------
    fig_slope, ax_slope = plt.subplots(figsize=(12, 10))
    
    height, width = dem.shape
    slope_map = np.zeros((height, width))
    
    for y in range(height):
        for x in range(width):
            max_slope = 0
            for dy, dx in [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)]:
                ny, nx = y + dy, x + dx
                if 0 <= ny < height and 0 <= nx < width:
                    # Get elevations with NaN handling
                    elev1 = dem_clean[y, x]
                    elev2 = dem_clean[ny, nx]
                    
                    # Only calculate slope if both points have valid data
                    if elev1 != 0 or elev2 != 0:  # Assuming 0 represents NaN/invalid data
                        elev_diff = abs(elev2 - elev1)
                        
                        dist = np.sqrt(dy*dy + dx*dx)
                        # Avoid division by zero
                        slope = np.degrees(np.arctan2(elev_diff, max(dist, 1e-10)))
                        max_slope = max(max_slope, slope)
            
            slope_map[y, x] = max_slope
    
    # Use a more robust colormap setup
    slope_cmap = plt.cm.YlOrRd
    slope_levels = [0, 10, 20, 30, 45, 60, 75, 90]
    slope_norm = BoundaryNorm(slope_levels, slope_cmap.N)
    
    slope_viz = ax_slope.imshow(slope_map, cmap=slope_cmap, norm=slope_norm, 
                         interpolation='nearest', alpha=0.7)
    
    if fast_path and len(fast_path) > 1:
        fast_y, fast_x = zip(*fast_path)
        ax_slope.plot(fast_x, fast_y, color='blue', linewidth=2, label='Distance-Optimized Path')
    
    if optimal_path and len(optimal_path) > 1:
        opt_y, opt_x = zip(*optimal_path)
        ax_slope.plot(opt_x, opt_y, color='magenta', linewidth=2, linestyle='--', label='Slope-Optimized Path')
    
    if local_start and local_end:
        ax_slope.scatter(local_start[1], local_start[0], c='g', s=100, marker='o', label='Start')
        ax_slope.scatter(local_end[1], local_end[0], c='r', s=100, marker='*', label='Goal')
    
    plt.colorbar(slope_viz, ax=ax_slope, label='Slope Angle (degrees)')
    ax_slope.set_title('Terrain Slope Map with Path Comparison')
    ax_slope.legend(loc='upper right')
    
    if buffer_region is not None and original_start is not None and original_end is not None:
        y_min, y_max, x_min, x_max = buffer_region
        region_text = f"Region: ({x_min}, {y_min}) to ({x_max}, {y_max})\n"
        region_text += f"Original Start: ({original_start[1]}, {original_start[0]}), End: ({original_end[1]}, {original_end[0]})"
        fig_slope.text(0.5, 0.01, region_text, ha='center', fontsize=10)
    
    slope_save_path = f"{base_save_path}_slope_map.png"
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.savefig(slope_save_path, dpi=200)
    plt.close(fig_slope)
    print(f"Slope map visualization saved to {slope_save_path}")
    
    # ------------------- Elevation Map Visualization -------------------
    fig_elev, ax_elev = plt.subplots(figsize=(12, 10))
    
    # Normalize the DEM data for visualization
    dem_valid = dem_clean[dem_clean != 0]  # Get only valid elevation values
    if len(dem_valid) > 0:
        min_val = np.min(dem_valid)
        max_val = np.max(dem_valid)
        
        if max_val > min_val:  # Avoid division by zero
            # Create normalized elevation map with proper handling of invalid values
            norm_dem = np.zeros_like(dem_clean)
            valid_mask = dem_clean != 0
            norm_dem[valid_mask] = (dem_clean[valid_mask] - min_val) / (max_val - min_val)
        else:
            norm_dem = np.zeros_like(dem_clean)
    else:
        norm_dem = np.zeros_like(dem_clean)
    
    # Create the elevation map visualization
    elevation_map = ax_elev.imshow(norm_dem, cmap='terrain', alpha=0.8)
    
    # The rest of your code for plotting paths, labels, etc.
    if fast_path and len(fast_path) > 1:
        fast_y, fast_x = zip(*fast_path)
        ax_elev.plot(fast_x, fast_y, color='blue', linewidth=2, label='Distance-Optimized Path')
    
    if optimal_path and len(optimal_path) > 1:
        opt_y, opt_x = zip(*optimal_path)
        ax_elev.plot(opt_x, opt_y, color='magenta', linewidth=2, linestyle='--', label='Slope-Optimized Path')
    
    if local_start and local_end:
        ax_elev.scatter(local_start[1], local_start[0], c='g', s=100, marker='o', label='Start')
        ax_elev.scatter(local_end[1], local_end[0], c='r', s=100, marker='*', label='Goal')
    
    plt.colorbar(elevation_map, ax=ax_elev, label='Normalized Elevation')
    ax_elev.set_title('Elevation Map with Path Comparison')
    ax_elev.legend(loc='upper right')
    
    if buffer_region is not None and original_start is not None and original_end is not None:
        y_min, y_max, x_min, x_max = buffer_region
        region_text = f"Region: ({x_min}, {y_min}) to ({x_max}, {y_max})\n"
        region_text += f"Original Start: ({original_start[1]}, {original_start[0]}), End: ({original_end[1]}, {original_end[0]})"
        fig_elev.text(0.5, 0.01, region_text, ha='center', fontsize=10)
    
    elev_save_path = f"{base_save_path}_elevation_map.png"
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.savefig(elev_save_path, dpi=200)
    plt.close(fig_elev)
    print(f"Elevation map visualization saved to {elev_save_path}")
    
    # ------------------- Path Profile Visualization -------------------
    # Only continue if we have paths to visualize
    if (fast_path and len(fast_path) > 1) or (optimal_path and len(optimal_path) > 1):
        fig_profile, ax_profile = plt.subplots(figsize=(12, 8))
        
        if fast_path and len(fast_path) > 1:
            # Safe extraction of elevation values along the path
            fast_elevations = []
            for pos in fast_path:
                if 0 <= pos[0] < dem.shape[0] and 0 <= pos[1] < dem.shape[1]:
                    elev = dem_clean[pos[0], pos[1]]
                    fast_elevations.append(elev)
                else:
                    fast_elevations.append(0)  # Use 0 for out-of-bounds positions
                    
            fast_distances = [0]
            fast_slopes = []
            
            for i in range(1, len(fast_path)):
                dx = fast_path[i][1] - fast_path[i-1][1]
                dy = fast_path[i][0] - fast_path[i-1][0]
                dist = np.sqrt(dx*dx + dy*dy)
                fast_distances.append(fast_distances[-1] + dist)
                
                # Safe elevation difference calculation
                elev1 = dem_clean[fast_path[i-1][0], fast_path[i-1][1]] if (0 <= fast_path[i-1][0] < dem.shape[0] and 0 <= fast_path[i-1][1] < dem.shape[1]) else 0
                elev2 = dem_clean[fast_path[i][0], fast_path[i][1]] if (0 <= fast_path[i][0] < dem.shape[0] and 0 <= fast_path[i][1] < dem.shape[1]) else 0
                elev_diff = abs(elev2 - elev1)
                
                # Avoid division by zero
                slope = np.degrees(np.arctan2(elev_diff, max(dist, 1e-10)))
                fast_slopes.append(slope)
            
            ax_profile.plot(fast_distances, fast_elevations, 'b-', label='Distance-Optimized Path')
            
            ax_slopes = ax_profile.twinx()
            ax_slopes.plot(fast_distances[1:], fast_slopes, 'b--', alpha=0.5, label='Slope (Distance-Opt)')
        
        # Similar changes for optimal_path section
        if optimal_path and len(optimal_path) > 1:
            # Apply the same fixes as above for the optimal path
            opt_elevations = []
            for pos in optimal_path:
                if 0 <= pos[0] < dem.shape[0] and 0 <= pos[1] < dem.shape[1]:
                    elev = dem_clean[pos[0], pos[1]]
                    opt_elevations.append(elev)
                else:
                    opt_elevations.append(0)
                    
            opt_distances = [0]
            opt_slopes = []
            
            for i in range(1, len(optimal_path)):
                dx = optimal_path[i][1] - optimal_path[i-1][1]
                dy = optimal_path[i][0] - optimal_path[i-1][0]
                dist = np.sqrt(dx*dx + dy*dy)
                opt_distances.append(opt_distances[-1] + dist)
                
                elev1 = dem_clean[optimal_path[i-1][0], optimal_path[i-1][1]] if (0 <= optimal_path[i-1][0] < dem.shape[0] and 0 <= optimal_path[i-1][1] < dem.shape[1]) else 0
                elev2 = dem_clean[optimal_path[i][0], optimal_path[i][1]] if (0 <= optimal_path[i][0] < dem.shape[0] and 0 <= optimal_path[i][1] < dem.shape[1]) else 0
                elev_diff = abs(elev2 - elev1)
                
                slope = np.degrees(np.arctan2(elev_diff, max(dist, 1e-10)))
                opt_slopes.append(slope)
            
            ax_profile.plot(opt_distances, opt_elevations, 'm-', label='Slope-Optimized Path')
            
            if 'ax_slopes' not in locals():
                ax_slopes = ax_profile.twinx()
            
            ax_slopes.plot(opt_distances[1:], opt_slopes, 'm--', alpha=0.5, label='Slope (Slope-Opt)')
        
        ax_profile.set_xlabel('Distance Along Path')
        ax_profile.set_ylabel('Elevation')
        ax_profile.set_title('Path Profiles: Elevation and Slope')
        ax_profile.grid(True)
        ax_profile.legend(loc='upper left')
        
        if 'ax_slopes' in locals():
            ax_slopes.set_ylabel('Slope (degrees)')
            ax_slopes.legend(loc='upper right')
            
            for slope_val in [15, 30, 45, 60]:
                ax_slopes.axhline(y=slope_val, color='gray', linestyle=':', alpha=0.5)
                ax_slopes.text(0, slope_val+1, f"{slope_val}°", fontsize=8, color='gray')
        
        profile_save_path = f"{base_save_path}_path_profile.png"
        plt.tight_layout()
        plt.savefig(profile_save_path, dpi=200)
        plt.close(fig_profile)
        print(f"Path profile visualization saved to {profile_save_path}")
    
    # The combined visualization would need the same fixes applied
    # I'll leave that part out for brevity, but you should apply the same pattern
    # of NaN handling and bounds checking throughout that section as well

In [81]:
def main():
    dem_path = '/kaggle/input/validation-set/Upper_Willow_Creek_BareEarth_Hillshade_1m_1.tif'
    
    print("=== Terrain Traversal Path Planner ===")
    print("\nThis program will find optimal paths for robot traversal between two points.")
    print("It calculates both the fastest theoretical path and the optimal path for a robot")
    print("considering terrain constraints like slopes and elevation changes.")
    
    try:
        print("\nSelect terrain resolution (higher value = faster but less detailed):")
        print("Recommended: 10-30 (10=detailed but slow, 30=faster but coarser)")
        downsample = int(input("Downsample factor [20]: ") or "20")
        if downsample < 1:
            downsample = 20
    except ValueError:
        downsample = 20
        
    print(f"Using downsample factor: {downsample}")
    
    try:
        print("\nHow much should the algorithm emphasize gentle slopes? (1-10)")
        print("1 = Barely consider slopes, 10 = Strongly avoid steep slopes")
        slope_emphasis = int(input("Slope emphasis [5]: ") or "5")
        if slope_emphasis < 1 or slope_emphasis > 10:
            slope_emphasis = 5
    except ValueError:
        slope_emphasis = 5
        
    print(f"Using slope emphasis factor: {slope_emphasis}")
    
    results = analyze_terrain_paths(dem_path, downsample_factor=downsample, buffer_size=75, 
                                    slope_emphasis=slope_emphasis)
    
    print("\n=== Path Comparison Summary ===")
    
    if results['fast_path']:
        print(f"\nDistance-Optimized Path:")
        print(f"- Processing time: {results['fast_time']:.2f} seconds")
        print(f"- Path length: {len(results['fast_path'])} steps ({results['fast_stats']['length']:.1f} units)")
        print(f"- Average slope: {results['fast_stats']['avg_slope']:.1f}°, Maximum slope: {results['fast_stats']['max_slope']:.1f}°")
        print(f"- Elevation changes: +{results['fast_stats']['elevation_gain']:.1f}, -{results['fast_stats']['elevation_loss']:.1f}")
    
    if results['optimal_path']:
        print(f"\nSlope-Optimized Path:")
        print(f"- Processing time: {results['optimal_time']:.2f} seconds")
        print(f"- Path length: {len(results['optimal_path'])} steps ({results['optimal_stats']['length']:.1f} units)")
        print(f"- Average slope: {results['optimal_stats']['avg_slope']:.1f}°, Maximum slope: {results['optimal_stats']['max_slope']:.1f}°")
        print(f"- Elevation changes: +{results['optimal_stats']['elevation_gain']:.1f}, -{results['optimal_stats']['elevation_loss']:.1f}")
    
    if results['fast_path'] and results['optimal_path']:
        length_diff = results['optimal_stats']['length'] - results['fast_stats']['length']
        length_percent = (length_diff / results['fast_stats']['length']) * 100
        slope_diff = results['fast_stats']['avg_slope'] - results['optimal_stats']['avg_slope']
        
        print(f"\nTrade-off Analysis:")
        print(f"- The slope-optimized path is {length_percent:.1f}% longer than the distance-optimized path")
        print(f"- The slope-optimized path has {slope_diff:.1f}° gentler average slopes")
    
    print("\nVisualization has been saved showing both paths.")
    print("The distance-optimized path (blue) represents the shortest route with any traversable slope.")
    print("The slope-optimized path (magenta) prioritizes gentler slopes even if it means a longer route.")

if __name__ == "__main__":
    main()

=== Terrain Traversal Path Planner ===

This program will find optimal paths for robot traversal between two points.
It calculates both the fastest theoretical path and the optimal path for a robot
considering terrain constraints like slopes and elevation changes.

Select terrain resolution (higher value = faster but less detailed):
Recommended: 10-30 (10=detailed but slow, 30=faster but coarser)


Downsample factor [20]:  30


Using downsample factor: 30

How much should the algorithm emphasize gentle slopes? (1-10)
1 = Barely consider slopes, 10 = Strongly avoid steep slopes


Slope emphasis [5]:  5


Using slope emphasis factor: 5


...

Coordinates (200, 400) are out of bounds!
Please choose different start coordinates.


Start X coordinate:  320
Start Y coordinate:  50
End X coordinate:  350
End Y coordinate:  10


Coordinates (350, 10) are out of bounds!
Please choose different end coordinates.


End X coordinate:  300
End Y coordinate:  60


Using Start: (320, 50), End: (300, 60)
Region dimensions: 135 x 109
Local start: (50, 95), Local end: (60, 75)

Finding distance-optimized path (minimal slope consideration)...
Path calculation took 0.73 seconds
Path found with 21 steps, length: 24.1 units
Average traversability: 1.3/100
Average slope: 84.6°, Maximum slope: 89.0°
Elevation gain: 261.8, loss: 238.8

Finding slope-optimized path (emphasizes gentle slopes)...
Path calculation took 1.63 seconds
Path found with 25 steps, length: 28.1 units
Average traversability: 1.5/100
Average slope: 64.3°, Maximum slope: 88.9°
Elevation gain: 123.5, loss: 100.5
Slope map visualization saved to terrain_paths_slope_map.png
Elevation map visualization saved to terrain_paths_elevation_map.png
Path profile visualization saved to terrain_paths_path_profile.png

=== Path Comparison Summary ===

Distance-Optimized Path:
- Processing time: 0.73 seconds
- Path length: 21 steps (24.1 units)
- Average slope: 84.6°, Maximum slope: 89.0°
- Elevation c